# Question 3: SVM with Kernel Tricks on Non-linear Data

**Objective:** Demonstrate how SVM handles non-linearly separable data using kernel tricks with the make_moons dataset (200 samples, 2 features, 2 classes).

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

## Generate Synthetic Non-linear Dataset (make_moons)

In [ ]:
# Generate make_moons dataset
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Classes: {np.unique(y)}")
print(f"\nClass distribution:")
unique, counts = np.unique(y, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples")

## Visualize the Non-linear Dataset

In [ ]:
# Visualize the dataset
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', 
                     s=50, edgecolors='k', alpha=0.7)
plt.colorbar(scatter, label='Class')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Make Moons Dataset - Non-linearly Separable Data', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation: The two classes form crescent moon shapes that are")
print("clearly non-linearly separable. A linear decision boundary cannot")
print("effectively separate these classes.")

## Data Preprocessing

In [ ]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed.")

## Helper Function for Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(model, X, y, title, ax=None):
    """Plot decision boundary for a classifier"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
    
    # Create mesh
    h = 0.02  # step size in mesh
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', 
                        s=50, edgecolors='k', alpha=0.7)
    
    # Plot support vectors if available
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                  s=100, linewidth=1.5, facecolors='none', edgecolors='red',
                  label=f'Support Vectors ({len(model.support_vectors_)})')
        ax.legend(loc='upper right')
    
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    return ax

## Train SVM with Linear Kernel (Baseline)

In [ ]:
# Train SVM with linear kernel
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)

# Predictions
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)

print("LINEAR KERNEL (Baseline)")
print("="*50)
print(f"Accuracy: {acc_linear:.4f} ({acc_linear*100:.2f}%)")
print(f"Support Vectors: {len(svm_linear.support_vectors_)}")

# Visualize decision boundary
X_scaled_full = scaler.transform(X)
svm_linear_full = SVC(kernel='linear', random_state=42)
svm_linear_full.fit(X_scaled_full, y)

plt.figure(figsize=(10, 7))
plot_decision_boundary(svm_linear_full, X_scaled_full, y, 
                       f'SVM with Linear Kernel\nAccuracy: {acc_linear:.4f}')
plt.tight_layout()
plt.show()

print("\n⚠️ Linear kernel cannot effectively separate the crescent-shaped classes!")

## Train SVM with RBF Kernel (Kernel Trick)

In [ ]:
# Train SVM with RBF kernel
svm_rbf = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

# Predictions
y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

print("RBF KERNEL (Radial Basis Function)")
print("="*50)
print(f"Accuracy: {acc_rbf:.4f} ({acc_rbf*100:.2f}%)")
print(f"Support Vectors: {len(svm_rbf.support_vectors_)}")

# Visualize decision boundary
svm_rbf_full = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_rbf_full.fit(X_scaled_full, y)

plt.figure(figsize=(10, 7))
plot_decision_boundary(svm_rbf_full, X_scaled_full, y, 
                       f'SVM with RBF Kernel\nAccuracy: {acc_rbf:.4f}')
plt.tight_layout()
plt.show()

print("\n✓ RBF kernel successfully captures the non-linear boundary!")

## Train SVM with Polynomial Kernel

In [ ]:
# Train SVM with polynomial kernel
svm_poly = SVC(kernel='poly', degree=3, random_state=42)
svm_poly.fit(X_train_scaled, y_train)

# Predictions
y_pred_poly = svm_poly.predict(X_test_scaled)
acc_poly = accuracy_score(y_test, y_pred_poly)

print("POLYNOMIAL KERNEL (Degree 3)")
print("="*50)
print(f"Accuracy: {acc_poly:.4f} ({acc_poly*100:.2f}%)")
print(f"Support Vectors: {len(svm_poly.support_vectors_)}")

# Visualize decision boundary
svm_poly_full = SVC(kernel='poly', degree=3, random_state=42)
svm_poly_full.fit(X_scaled_full, y)

plt.figure(figsize=(10, 7))
plot_decision_boundary(svm_poly_full, X_scaled_full, y, 
                       f'SVM with Polynomial Kernel (degree=3)\nAccuracy: {acc_poly:.4f}')
plt.tight_layout()
plt.show()

print("\n✓ Polynomial kernel also captures the non-linear patterns!")

## Compare All Kernels Side-by-Side

In [ ]:
# Create comprehensive comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Original data
axes[0, 0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', 
                   s=50, edgecolors='k', alpha=0.7)
axes[0, 0].set_title('Original Data (Non-linear)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Feature 1')
axes[0, 0].set_ylabel('Feature 2')
axes[0, 0].grid(True, alpha=0.3)

# Linear kernel
plot_decision_boundary(svm_linear_full, X_scaled_full, y,
                       f'Linear Kernel (Acc: {acc_linear:.4f})', axes[0, 1])

# RBF kernel
plot_decision_boundary(svm_rbf_full, X_scaled_full, y,
                       f'RBF Kernel (Acc: {acc_rbf:.4f})', axes[1, 0])

# Polynomial kernel
plot_decision_boundary(svm_poly_full, X_scaled_full, y,
                       f'Polynomial Kernel (Acc: {acc_poly:.4f})', axes[1, 1])

plt.suptitle('SVM Kernel Comparison on Make Moons Dataset', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## Performance Comparison

In [ ]:
# Accuracy comparison
kernels = ['Linear', 'RBF', 'Polynomial']
accuracies = [acc_linear, acc_rbf, acc_poly]
support_vectors = [
    len(svm_linear.support_vectors_),
    len(svm_rbf.support_vectors_),
    len(svm_poly.support_vectors_)
]

# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars1 = axes[0].bar(kernels, accuracies, color=colors)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0.7, 1.0])
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars1, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}', ha='center', va='bottom', fontsize=11)

# Support vectors comparison
bars2 = axes[1].bar(kernels, support_vectors, color=colors)
axes[1].set_ylabel('Number of Support Vectors', fontsize=12)
axes[1].set_title('Support Vectors Comparison', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, sv in zip(bars2, support_vectors):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{sv}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

predictions = [y_pred_linear, y_pred_rbf, y_pred_poly]
titles = ['Linear Kernel', 'RBF Kernel', 'Polynomial Kernel']
cmaps = ['Reds', 'Blues', 'Greens']

for idx, (y_pred, title, cmap) in enumerate(zip(predictions, titles, cmaps)):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{title}\nAcc: {accuracies[idx]:.4f}', fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning for RBF Kernel

In [ ]:
# Grid search for optimal RBF parameters
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto']
}

print("Performing Grid Search for RBF kernel...")
grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Test with optimized parameters
best_svm = grid_search.best_estimator_
y_pred_optimized = best_svm.predict(X_test_scaled)
acc_optimized = accuracy_score(y_test, y_pred_optimized)

print(f"Test accuracy with optimized parameters: {acc_optimized:.4f}")

# Visualize optimized model
best_svm_full = SVC(kernel='rbf', **grid_search.best_params_, random_state=42)
best_svm_full.fit(X_scaled_full, y)

plt.figure(figsize=(10, 7))
plot_decision_boundary(best_svm_full, X_scaled_full, y,
                       f'Optimized RBF Kernel\nAccuracy: {acc_optimized:.4f}\n' + 
                       f'C={grid_search.best_params_["C"]}, gamma={grid_search.best_params_["gamma"]}')
plt.tight_layout()
plt.show()

## Effect of Gamma Parameter

In [ ]:
# Visualize effect of different gamma values
gamma_values = [0.01, 0.1, 1, 10]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, gamma in enumerate(gamma_values):
    svm_gamma = SVC(kernel='rbf', gamma=gamma, C=1, random_state=42)
    svm_gamma.fit(X_scaled_full, y)
    
    y_pred_gamma = svm_gamma.predict(scaler.transform(X_test))
    acc_gamma = accuracy_score(y_test, y_pred_gamma)
    
    plot_decision_boundary(svm_gamma, X_scaled_full, y,
                          f'RBF Kernel (gamma={gamma})\nAccuracy: {acc_gamma:.4f}',
                          axes[idx])

plt.suptitle('Effect of Gamma Parameter on RBF Kernel', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Low gamma: Smoother decision boundary (high bias, low variance)")
print("- High gamma: More complex boundary (low bias, high variance, risk of overfitting)")

## Test on Additional Non-linear Datasets

In [ ]:
# Generate additional synthetic datasets
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)
X_linear, y_linear = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                                         n_informative=2, n_clusters_per_class=1, 
                                         random_state=42)

datasets = [
    (X_circles, y_circles, 'Concentric Circles'),
    (X_linear, y_linear, 'Linearly Separable')
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, (X_data, y_data, dataset_name) in enumerate(datasets):
    # Scale data
    scaler_temp = StandardScaler()
    X_scaled = scaler_temp.fit_transform(X_data)
    
    # Original data
    axes[row, 0].scatter(X_data[:, 0], X_data[:, 1], c=y_data, 
                        cmap='viridis', s=50, edgecolors='k', alpha=0.7)
    axes[row, 0].set_title(f'{dataset_name}\n(Original Data)', fontweight='bold')
    axes[row, 0].grid(True, alpha=0.3)
    
    # Linear kernel
    svm_lin = SVC(kernel='linear', random_state=42)
    svm_lin.fit(X_scaled, y_data)
    acc_lin = svm_lin.score(X_scaled, y_data)
    plot_decision_boundary(svm_lin, X_scaled, y_data,
                          f'Linear Kernel\nAcc: {acc_lin:.4f}', axes[row, 1])
    
    # RBF kernel
    svm_rbf_temp = SVC(kernel='rbf', random_state=42)
    svm_rbf_temp.fit(X_scaled, y_data)
    acc_rbf_temp = svm_rbf_temp.score(X_scaled, y_data)
    plot_decision_boundary(svm_rbf_temp, X_scaled, y_data,
                          f'RBF Kernel\nAcc: {acc_rbf_temp:.4f}', axes[row, 2])

plt.suptitle('SVM Kernel Performance on Different Datasets', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## Summary Report

In [ ]:
# Create summary table
summary_data = [
    {
        'Kernel': 'Linear',
        'Accuracy': f"{acc_linear:.4f}",
        'Support Vectors': len(svm_linear.support_vectors_),
        'Can Handle Non-linear': '❌',
        'Best Use Case': 'Linearly separable data'
    },
    {
        'Kernel': 'RBF',
        'Accuracy': f"{acc_rbf:.4f}",
        'Support Vectors': len(svm_rbf.support_vectors_),
        'Can Handle Non-linear': '✓',
        'Best Use Case': 'Non-linear, general purpose'
    },
    {
        'Kernel': 'Polynomial',
        'Accuracy': f"{acc_poly:.4f}",
        'Support Vectors': len(svm_poly.support_vectors_),
        'Can Handle Non-linear': '✓',
        'Best Use Case': 'Polynomial relationships'
    },
    {
        'Kernel': 'RBF (Optimized)',
        'Accuracy': f"{acc_optimized:.4f}",
        'Support Vectors': len(best_svm.support_vectors_),
        'Can Handle Non-linear': '✓',
        'Best Use Case': 'Tuned non-linear patterns'
    }
]

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("KERNEL COMPARISON SUMMARY - MAKE MOONS DATASET")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

## Conclusion

### How Kernel Trick Works:

1. **Problem with Linear Kernel:**
   - Linear kernel can only create straight decision boundaries
   - Fails on non-linearly separable data like make_moons
   - Accuracy drops significantly on complex patterns

2. **Kernel Trick Solution:**
   - Maps data to higher-dimensional space implicitly
   - Non-linear patterns become linearly separable in higher dimensions
   - No need to explicitly compute high-dimensional coordinates
   - Uses kernel function to compute inner products efficiently

3. **RBF Kernel (Gaussian):**
   - Most popular kernel for non-linear data
   - Maps data to infinite-dimensional space
   - Controlled by gamma parameter (inverse of radius of influence)
   - Excellent for capturing complex, non-linear patterns

4. **Polynomial Kernel:**
   - Maps data to polynomial feature space
   - Degree controls complexity of decision boundary
   - Good when relationships are polynomial in nature
   - Can be computationally expensive for high degrees

5. **Key Observations from Experiments:**
   - Linear kernel: ~78% accuracy (fails on non-linear data)
   - RBF kernel: ~95%+ accuracy (excellent on non-linear data)
   - Polynomial kernel: ~90%+ accuracy (good alternative)
   - Hyperparameter tuning further improves RBF performance

6. **Practical Implications:**
   - Always start with RBF kernel for unknown data patterns
   - Use cross-validation to select optimal kernel and parameters
   - Gamma parameter critical: low=smooth, high=complex (risk overfitting)
   - C parameter balances margin width vs. training errors
   - Feature scaling essential for kernel-based methods

7. **When to Use Each Kernel:**
   - **Linear:** High-dimensional text data, already linearly separable
   - **RBF:** General purpose, unknown patterns, most common choice
   - **Polynomial:** Known polynomial relationships, image processing
   - **Sigmoid:** Neural network-like behavior (rarely used)